In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(" All imports done!")

 All imports done!


In [3]:
DATA_DIR = Path("../data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

## 3. Text Preprocessor

Custom class that handles:
- Stripping MCQ-style prefixes (*"Pick the best answer: "*)
- Stripping suffixes (*"from the following options."*)
- Normalising whitespace and optional lowercasing

### 3.1 Class Definition

In [8]:
class TextPreprocessor:
    def __init__(self, lowercase=True):
        
        self.lowercase = lowercase
        
        # Common question prefixes (MCQ questions have these patterns)
        self.prefixes = [
            "pick the best possible answer",
            "pick the best answer",
            "identify the best answer",
            "identify the correct",
            "determine the best",
            "choose the best",
            "select the best",
            "which of the following",
        ]
        
        # Common suffixes are 
        self.suffixes = [
            "among the following options",
            "among the listed options",
            "from the following options",
            "from the following choices",
            "from the given options",
        ]
    
    def remove_prefix(self, text):
        """
        Remove common question prefixes
        Example: 'Pick the best answer: What is X?' → 'What is X?'
        """
        text_lower = text.lower().strip()
        
        for prefix in self.prefixes:
            if text_lower.startswith(prefix):
                # Remove prefix and following colon/space
                text = text[len(prefix):].strip()
                text = re.sub(r'^[:?\s]+', '', text)
                break
        
        return text
    
    def remove_suffix(self, text):
        """
        Remove common suffixes
        Example: 'What is X? from the following options.' → 'What is X?'
        """
        text_lower = text.lower().strip()
        
        for suffix in self.suffixes:
            if text_lower.endswith(suffix):
                # Remove suffix
                text = text[:-len(suffix)].strip()
                # Remove trailing punctuation
                text = re.sub(r'[\?\.:;\s]+$', '', text)
                break
        
        return text
    
    def clean_whitespace(self, text):
        text = re.sub(r'\s+', ' ', text)  # Multiple spaces to single
        text = text.strip()  # Remove leading/trailing
        return text
    
    def normalize_text(self, text):
        """
        Apply all cleaning steps in order
        Remove prefix --> Remove suffix --> Clean whitespace --> Lowercase
        """
        text = self.remove_prefix(text)
        text = self.remove_suffix(text)
        text = self.clean_whitespace(text)
        
        if self.lowercase:
            text = text.lower()
        
        return text

preprocessor = TextPreprocessor()
print("TextPreprocessor created ")

TextPreprocessor created 


In [9]:
train_clean = train_df.copy()
test_clean  = test_df.copy()

# clean prompts
train_clean['prompt'] = train_clean['prompt'].apply(preprocessor.normalize_text)
test_clean['prompt']  = test_clean['prompt'].apply(preprocessor.normalize_text)

# clean all five options
for col in ['A', 'B', 'C', 'D', 'E']:
    train_clean[col] = train_clean[col].apply(preprocessor.normalize_text)
    test_clean[col]  = test_clean[col].apply(preprocessor.normalize_text)

print('Preprocessing done.')
print(f'\nBefore: {train_df.iloc[0]["prompt"]}')
print(f'After : {train_clean.iloc[0]["prompt"]}')

Preprocessing done.

Before: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
After : what is martin heidegger's view on the relationship between time and human existence? among the listed options.
